# SwinV2 Image Classification — DIMER E2E tutorial

**Profile:** `E2E` · **Notebook spec:** 1.0  
**Pipeline:** `kurtvalcorza/swin-classification-pipeline`  
**Capability:** image-folder validation → supervised SwinV2 fine-tuning → persisted artifact verification → new-image inference.

This notebook exercises the pipeline release's pinned validator and finetuner workers. Fine-tuning uses gradient updates; it is not zero-shot or in-context learning. The default dataset is deterministic synthetic tutorial data, so its metrics are sanity evidence only.

You will verify immutable source/model provenance, validate the sample, run the real worker CLIs, inspect task metrics and a majority baseline, reload the exported `safetensors` model for new-data inference, and export machine-readable prediction/provenance.

**Not demonstrated:** ImageNet benchmark reproduction, calibrated probabilities, CPU training, production fitness, detection, or segmentation.


## Prerequisites

Use Python 3.11+ with an NVIDIA GPU exposed as `cuda:0`. Network access is needed only to clone sources and stage the pinned checkpoint. The worker itself consumes local model bytes. The repository's formal accelerator qualification is narrower than “any Colab GPU”; a successful run elsewhere is execution evidence for that runtime, not an extension of formal qualification.

The default path does not implement BYOD folder upload (`DAT7` SHOULD deviation recorded in `tutorials/README.md`).


The pinned validator and finetuner source repositories are private. Provide a Colab Secret named `GITHUB_TOKEN` (or a `GITHUB_TOKEN` environment variable outside Colab) with read access to those repositories. Authentication is passed to Git through an ephemeral HTTP header; the token is not printed, embedded in a URL, or stored in repository config.

## 1. Bootstrap immutable sources and runtime


In [ ]:
from __future__ import annotations
import base64, hashlib, json, os, platform, random, shutil, subprocess, sys
from pathlib import Path

PIPELINE_REF="f68176e95577d6106538a4d8f8ea592a6d5ca848"
W=Path("/content/dimer-swin-classification"); P=W/"pipeline"; V=W/"validator"; F=W/"finetuner"
def run(cmd,cwd=None,env=None):
    cmd=[str(x) for x in cmd]; print("+"," ".join(cmd)); subprocess.run(cmd,cwd=cwd,check=True,env=env)

def private_github_token():
    token=os.environ.get("GITHUB_TOKEN","").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        token=(userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        token=""
    if not token:
        raise RuntimeError("Private validator/finetuner source requires a GITHUB_TOKEN Colab Secret or environment variable with read access.")
    return token

def private_git_env(token):
    credential=base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env=os.environ.copy()
    env.update({"GIT_TERMINAL_PROMPT":"0","GIT_CONFIG_COUNT":"1","GIT_CONFIG_KEY_0":"http.https://github.com/.extraHeader","GIT_CONFIG_VALUE_0":f"Authorization: Basic {credential}"})
    return env

W.mkdir(parents=True,exist_ok=True)
for url,ref,dst in [("https://github.com/kurtvalcorza/swin-classification-pipeline.git",PIPELINE_REF,P)]:
    if not dst.exists(): run(["git","clone","--filter=blob:none",url,dst])
    run(["git","checkout","--detach",ref],dst)
pm=json.loads((P/"pipeline-manifest.json").read_text())
vr=json.loads((P/"release/worker-release-validator.json").read_text())
fr=json.loads((P/"release/worker-release-finetuner.json").read_text())
PRIVATE_TOKEN=private_github_token()
PRIVATE_GIT_ENV=private_git_env(PRIVATE_TOKEN)
for url,ref,dst in [
 ("https://github.com/kurtvalcorza/swin-classification-dataset-validator.git",vr["sourceRevision"],V),
 ("https://github.com/kurtvalcorza/swin-classification-finetuner.git",fr["sourceRevision"],F)]:
    if not dst.exists(): run(["git","clone","--filter=blob:none",url,dst],env=PRIVATE_GIT_ENV)
    run(["git","checkout","--detach",ref],dst,env=PRIVATE_GIT_ENV)
del PRIVATE_TOKEN, PRIVATE_GIT_ENV

run([sys.executable,"-m","pip","install","-q","timm==1.0.28","huggingface_hub==1.29.0","safetensors==0.8.0","pillow==12.3.0"])
run([sys.executable,"-m","pip","install","-q","--no-deps","-e",V])
run([sys.executable,"-m","pip","install","-q","--no-deps","-e",F])

import timm, torch, torchvision
from PIL import Image, ImageDraw
runtime={"python":platform.python_version(),"torch":torch.__version__,"torchvision":torchvision.__version__,
         "timm":timm.__version__,"cuda":torch.version.cuda,
         "device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(json.dumps({"sources":{"pipeline":PIPELINE_REF,"validator":vr["sourceRevision"],"finetuner":fr["sourceRevision"]},"runtime":runtime},indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Colab choose a GPU runtime, then Run all.")


## 2. Resolve the pinned model and create the default sample

The pipeline allowlists SwinV2 Tiny/Small ImageNet-1K weights. The default Tiny checkpoint is resolved from the finetuner catalog and cross-checked against pipeline provenance. The sample preserves explicit `train/` and `val/` ownership and two case-sensitive classes.


In [ ]:
MODEL_KEY="swinv2-tiny-window8-256-ms-in1k" # @param ["swinv2-tiny-window8-256-ms-in1k","swinv2-small-window8-256-ms-in1k"]
catalog=json.loads((F/"catalog/base-model-catalog.json").read_text()); entry=catalog["entries"][MODEL_KEY]
prov=json.loads((P/"provenance/open-weights.json").read_text())
pe=next(x for x in prov["operationalWeights"] if x["catalogKey"]==MODEL_KEY)
assert entry["source"]["revision"]==pe["modelDescriptorVersion"]
assert entry["source"]["files"][0]["digest"]=="sha256:"+pe["sha256"]
identity={"modelKey":MODEL_KEY,"modelDescriptorId":entry["modelDescriptorId"],"repo":entry["source"]["repoId"],
          "revision":entry["source"]["revision"],"digest":entry["source"]["files"][0]["digest"]}
print(json.dumps(identity,indent=2))

D=W/"sample-data"
if D.exists(): shutil.rmtree(D)
rng=random.Random(20260910)
for split,n in {"train":8,"val":4}.items():
    for name in ("cool","warm"):
        out=D/split/name; out.mkdir(parents=True,exist_ok=True)
        for i in range(n):
            im=Image.new("RGB",(256,256),(35,70,190) if name=="cool" else (190,65,35)); dr=ImageDraw.Draw(im)
            j=rng.randint(-15,15)
            if name=="cool": dr.rectangle((60+j,60,196+j,196),outline=(230,240,255),width=10)
            else: dr.ellipse((60+j,60,196+j,196),outline=(255,240,220),width=10)
            im.save(out/f"{name}-{i:02d}.png")
print("Synthetic samples:",sum(1 for _ in D.rglob("*.png")))


## 3. Validate through the real dataset worker

The validator freezes logical sample identity, label semantics and split assignments. Tutorial authority digests below are deterministic caller-supplied stand-ins; they are not production DIMER admission/security records.


In [ ]:
def dj(x): return "sha256:"+hashlib.sha256(json.dumps(x,sort_keys=True,separators=(",",":")).encode()).hexdigest()
auth={"job":dj({"kind":"tutorial-job","task":pm["taskProfile"]}),
      "admission":dj({"kind":"tutorial-admission"}),"security":dj({"kind":"tutorial-security","networkDuringRunning":"DENY"})}
H=W/"validated"
if H.exists(): shutil.rmtree(H)
run(["swin-classification-validate",D,H,"--job-id","tutorial-classification","--attempt-id","attempt-1",
     "--worker-release-digest",pm["validatorWorkerReleaseDigest"],"--effective-job-spec-digest",auth["job"],
     "--admission-record-digest",auth["admission"],"--security-grant-digest",auth["security"]])
vresult=json.loads((H/"result.json").read_text()); print(json.dumps(vresult,indent=2))
if vresult.get("state")!="SUCCEEDED": raise RuntimeError("Validation failed; inspect result.json.")


## 4. Stage and SHA-256 verify the selected upstream checkpoint

Acquisition uses the immutable repository revision in the catalog. The worker will later load only the staged local file. Digest equality establishes byte identity with the catalogued file; it is not publisher-authenticity proof.


In [ ]:
from huggingface_hub import hf_hub_download
src=entry["source"]; fe=src["files"][0]; R=W/"weights"
downloaded=Path(hf_hub_download(repo_id=src["repoId"],filename=fe["path"],revision=src["revision"]))
target=R/MODEL_KEY/fe["path"]; target.parent.mkdir(parents=True,exist_ok=True); shutil.copyfile(downloaded,target)
h=hashlib.sha256()
with target.open("rb") as f:
    for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
observed="sha256:"+h.hexdigest(); print("expected",fe["digest"],"\nobserved",observed)
if observed!=fe["digest"]: target.unlink(missing_ok=True); raise RuntimeError("Checkpoint digest mismatch.")


## 5. Fine-tune, publish, reload, and evaluate

`swin-classification-train` is the pinned production-facing worker CLI. It consumes the validator handoff without re-splitting, performs supervised fine-tuning, writes a content-addressed artifact, reloads the persisted model, and evaluates the frozen validation split.

The one-epoch default is intentionally small. Reported metrics are tutorial metrics from one synthetic holdout, with no dispersion estimate.


In [ ]:
O=W/"training-output"
if O.exists(): shutil.rmtree(O)
EPOCHS=1 # @param {type:"integer"}
BATCH_SIZE=4 # @param {type:"integer"}
SEED=20260910 # @param {type:"integer"}
run(["swin-classification-train",D,H,R,F/"catalog/base-model-catalog.json",O,
     "--model-key",MODEL_KEY,"--expected-accelerator","cuda:0","--job-id","tutorial-classification","--attempt-id","attempt-1",
     "--worker-release-digest",pm["finetunerWorkerReleaseDigest"],"--effective-job-spec-digest",auth["job"],
     "--admission-record-digest",auth["admission"],"--security-grant-digest",auth["security"],
     "--epochs",EPOCHS,"--batch-size",BATCH_SIZE,"--seed",SEED])
result=json.loads((O/"result.json").read_text()); evaluation=json.loads((O/"evaluation-report.json").read_text())
print(json.dumps({"result":result,"evaluation":evaluation},indent=2))
if result.get("state")!="SUCCEEDED": raise RuntimeError("Fine-tuning failed; inspect terminal manifests.")


## 6. Baseline and new-data inference from the persisted artifact

The validation sample is balanced, so majority-class accuracy is 0.5. For a separate new image, the exported artifact is reconstructed from disk. The decision rule is `argmax(logits)`. Softmax outputs are **uncalibrated class scores**, not calibrated probabilities.


In [ ]:
metrics={x["id"]:x["value"] for x in evaluation["metrics"]}
comparison={"estimation":"single frozen synthetic validation split","majorityBaselineAccuracy":0.5,
            "modelAccuracy":metrics.get("core.metric.classification.accuracy"),
            "crossEntropy":metrics.get("org.valcorza.metric.classification.cross-entropy")}
print(json.dumps(comparison,indent=2))

from safetensors.torch import load_file
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import torch.nn.functional as TF

gen=O/"artifact/generations"/(O/"artifact/CURRENT").read_text().strip()
am=json.loads((gen/"artifact-manifest.json").read_text()); mc=json.loads((gen/"model-config.json").read_text())
m=timm.create_model(mc["timmModelName"],pretrained=False,num_classes=mc["numClasses"])
m.load_state_dict(load_file(gen/"model.safetensors"),strict=True); m=m.to("cuda:0").eval()
im=Image.new("RGB",(256,256),(190,65,35)); ImageDraw.Draw(im).ellipse((68,68,188,188),outline=(255,240,220),width=12)
prep=transforms.Compose([transforms.Resize((256,256),interpolation=InterpolationMode.BICUBIC,antialias=True),
                         transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
with torch.no_grad(): scores=TF.softmax(m(prep(im).unsqueeze(0).to("cuda:0")),dim=1)[0].cpu()
classes=mc["classNames"]; k=int(scores.argmax())
prediction={"input":"synthetic-new-warm","decisionRule":"argmax(logits)","predictedClass":classes[k],
            "uncalibratedSoftmaxScores":{c:float(scores[i]) for i,c in enumerate(classes)}}
print(json.dumps(prediction,indent=2))


## 7. Export machine-readable results and provenance


In [ ]:
X=W/"exports"; X.mkdir(exist_ok=True)
px={"notebookProfile":"E2E","notebookSpec":"1.0","pipelineRevision":PIPELINE_REF,"pipelineId":pm["pipelineId"],
    "validatorRevision":vr["sourceRevision"],"finetunerRevision":fr["sourceRevision"],"model":identity,"runtime":runtime,
    "tutorialData":{"type":"deterministic synthetic","seed":20260910},"training":{"epochs":EPOCHS,"batchSize":BATCH_SIZE,"seed":SEED},
    "evaluation":comparison,"artifactBundleDigest":am["bundleDigest"]}
(X/"prediction.json").write_text(json.dumps(prediction,indent=2)+"\n")
(X/"provenance.json").write_text(json.dumps(px,indent=2)+"\n")
print(*(str(p) for p in sorted(X.glob("*.json"))),sep="\n")


## Interpretation and limits

A successful top-to-bottom run proves, for the recorded runtime, that the pinned validator accepted the sample, the checkpoint bytes matched the immutable catalog identity, the real finetuner completed gradient training and persisted reload/evaluation, and the exported artifact could be reconstructed for a new-image prediction.

It does **not** prove benchmark reproduction, real-domain generalization, calibration, fairness, robustness, safety, or production fitness. Formal accelerator qualification remains limited to the repository's recorded evidence. Before release, record a clean-runtime execution for this notebook revision; static notebook checks alone are not execution evidence.
